# Percobaan 4 — Klasifikasi Bunga (Mawar, Lili, Anggrek)

**Apa yang beda dari Percobaan 1-3?**

Percobaan 1-3 cuma pakai fitur **tekstur** (GLCM) dari citra grayscale. Padahal mawar, lili,
dan anggrek itu paling gampang dibedakan lewat **warna**-nya, bukan cuma teksturnya. Jadi di
Percobaan 4 ini kita gabungin:

1. **CLAHE** (Contrast Limited Adaptive Histogram Equalization) — versi "pintar" dari histogram
   equalization yang dipakai di Percobaan 2. Bedanya, CLAHE bekerja per-tile (lokal per area kecil),
   jadi kontras yang ditingkatkan lebih merata dan nggak gampang bikin noise menumpuk di satu titik.
2. **Fitur tekstur GLCM** (0°, 45°, 90°, 135°) — sama kayak sebelumnya, dihitung dari citra
   grayscale yang udah di-CLAHE.
3. **Fitur warna Histogram HSV** (Hue, Saturation, Value) — fitur baru. Hue itu yang paling
   "ngomong" soal warna bunga (merah/ungu/kuning, dst), jadi seharusnya sangat membantu misahin
   3 kelas ini.

Alur lainnya (seleksi fitur korelasi, split data, standardisasi, RF/SVM/KNN, confusion matrix)
tetap mengikuti pola Percobaan 1-3 biar hasilnya bisa dibandingkan apple-to-apple.

In [ ]:
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay)
import seaborn as sns

## 1. Load Dataset

In [ ]:
data = []
labels = []
file_name = []

IMG_SIZE = (128, 128)

for sub_folder in os.listdir("dataset"):
    sub_folder_files = os.listdir(os.path.join("dataset", sub_folder))
    for i, filename in enumerate(sub_folder_files):
        img_path = os.path.join("dataset", sub_folder, filename)
        img = cv.imread(img_path)

        if img is None:
            continue

        data.append(img)
        labels.append(sub_folder)
        file_name.append(f"{sub_folder}_{i+1}.jpg")

print(f"Total data: {len(data)}")
print(f"Kelas: {sorted(set(labels))}")

## 2. Preprocessing

Karena sekarang kita butuh **dua versi** citra per gambar:
- versi **grayscale + CLAHE** → buat fitur tekstur (GLCM)
- versi **warna (resize doang)** → buat fitur warna (Histogram HSV)

In [ ]:
TARGET_SIZE = (128, 128)

def resize_grayscale(image, target_size=TARGET_SIZE):
    resized = cv.resize(image, target_size)

    if len(resized.shape) == 3:
        gray = cv.cvtColor(resized, cv.COLOR_BGR2GRAY)
    else:
        gray = resized

    return gray.astype(np.uint8)

def resize_color(image, target_size=TARGET_SIZE):
    # Tetap BGR (warna asli), khusus dipakai buat ekstraksi fitur warna HSV
    resized = cv.resize(image, target_size)
    return resized

def apply_clahe(gray_img, clip_limit=2.0, tile_grid_size=(8, 8)):
    # CLAHE = Contrast Limited Adaptive Histogram Equalization.
    # Beda sama equalizeHist() biasa (Percobaan 2) yang ngitung histogram dari SELURUH citra,
    # CLAHE membagi citra jadi tile-tile kecil (default 8x8) terus equalize tiap tile-nya
    # secara lokal -> kontras tekstur jadi lebih nendang tapi tetap stabil (clip_limit
    # mencegah noise ikut ter-amplifikasi).
    clahe = cv.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray_img)

In [ ]:
def prepro_gray(image):
    # Resize + Grayscale + CLAHE -> dipakai buat fitur tekstur (GLCM)
    img = resize_grayscale(image)
    img = apply_clahe(img)
    return img

def prepro_color(image):
    # Resize aja, tetap warna -> dipakai buat fitur warna (Histogram HSV)
    img = resize_color(image)
    return img

## 3. Jalankan Preprocessing + Visualisasi Hasil

In [ ]:
def percobaan4(img):
    hasil_gray  = prepro_gray(img)
    hasil_color = prepro_color(img)
    return hasil_gray, hasil_color

hasil_prepro = [percobaan4(img) for img in data]
dataPreprocessedGray  = [h[0] for h in hasil_prepro]
dataPreprocessedColor = [h[1] for h in hasil_prepro]

unique_labels = sorted(set(labels))

for label in unique_labels:
    idxs = [j for j, l in enumerate(labels) if l == label]

    fig, axs = plt.subplots(7, 10, figsize=(15, 10))
    fig.suptitle(f'{label} (grayscale + CLAHE)', fontsize=16)

    for k in range(min(70, len(idxs))):

        row = k // 10
        col = k % 10

        ax = axs[row][col]

        ax.imshow(dataPreprocessedGray[idxs[k]], cmap='gray')
        ax.axis('off')

    for k in range(len(idxs), 70):
        row = k // 10
        col = k % 10
        axs[row][col].axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

## 4. Fungsi Ekstraksi Fitur Tekstur (GLCM) — sama seperti Percobaan 1-3

In [ ]:
def glcm(image, derajat):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Invalid angle. It should be one of the following: 0, 45, 90, 135.")

    glcm = graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)
    return glcm

In [ ]:
def correlation_feat(matriks):
    return graycoprops(matriks, 'correlation')[0, 0]

def dissimilarity(matriks):
    return graycoprops(matriks, 'dissimilarity')[0, 0]

def homogenity(matriks):
    return graycoprops(matriks, 'homogeneity')[0, 0]

def contrast(matriks):
    return graycoprops(matriks, 'contrast')[0, 0]

def ASM(matriks):
    return graycoprops(matriks, 'ASM')[0, 0]

def energy(matriks):
    return graycoprops(matriks, 'energy')[0, 0]

def entropyGlcm(matriks):
    return entropy(matriks.ravel())

## 5. Fungsi Ekstraksi Fitur Warna (Histogram HSV) — BARU di Percobaan 4

- **Hue** → rentang warna (merah, ungu, kuning, dst) → paling kuat buat bedain mawar/lili/anggrek
- **Saturation** → kepekatan/intensitas warnanya
- **Value** → tingkat terang-gelapnya

Tiap channel dipecah jadi 8 bin histogram, lalu dinormalisasi (supaya nilainya nggak bias
karena ukuran citra).

In [ ]:
def color_histogram_hsv(image_bgr, bins=8):
    hsv = cv.cvtColor(image_bgr, cv.COLOR_BGR2HSV)

    hist_h = cv.calcHist([hsv], [0], None, [bins], [0, 180])
    hist_s = cv.calcHist([hsv], [1], None, [bins], [0, 256])
    hist_v = cv.calcHist([hsv], [2], None, [bins], [0, 256])

    hist_h = cv.normalize(hist_h, hist_h).flatten()
    hist_s = cv.normalize(hist_s, hist_s).flatten()
    hist_v = cv.normalize(hist_v, hist_v).flatten()

    return hist_h, hist_s, hist_v

## 6. Jalankan Ekstraksi Fitur Tekstur (GLCM)

In [ ]:
Derajat0 = []
Derajat45 = []
Derajat90 = []
Derajat135 = []

for i in range(len(dataPreprocessedGray)):
    D0   = glcm(dataPreprocessedGray[i], 0)
    D45  = glcm(dataPreprocessedGray[i], 45)
    D90  = glcm(dataPreprocessedGray[i], 90)
    D135 = glcm(dataPreprocessedGray[i], 135)
    Derajat0.append(D0)
    Derajat45.append(D45)
    Derajat90.append(D90)
    Derajat135.append(D135)

Kontras0, Kontras45, Kontras90, Kontras135         = [], [], [], []
dissimilarity0, dissimilarity45, dissimilarity90, dissimilarity135 = [], [], [], []
homogenity0, homogenity45, homogenity90, homogenity135 = [], [], [], []
entropy0, entropy45, entropy90, entropy135         = [], [], [], []
ASM0, ASM45, ASM90, ASM135                         = [], [], [], []
energy0, energy45, energy90, energy135             = [], [], [], []
correlation0, correlation45, correlation90, correlation135 = [], [], [], []

for i in range(len(dataPreprocessedGray)):
    Kontras0.append(contrast(Derajat0[i]))
    Kontras45.append(contrast(Derajat45[i]))
    Kontras90.append(contrast(Derajat90[i]))
    Kontras135.append(contrast(Derajat135[i]))

    dissimilarity0.append(dissimilarity(Derajat0[i]))
    dissimilarity45.append(dissimilarity(Derajat45[i]))
    dissimilarity90.append(dissimilarity(Derajat90[i]))
    dissimilarity135.append(dissimilarity(Derajat135[i]))

    homogenity0.append(homogenity(Derajat0[i]))
    homogenity45.append(homogenity(Derajat45[i]))
    homogenity90.append(homogenity(Derajat90[i]))
    homogenity135.append(homogenity(Derajat135[i]))

    entropy0.append(entropyGlcm(Derajat0[i]))
    entropy45.append(entropyGlcm(Derajat45[i]))
    entropy90.append(entropyGlcm(Derajat90[i]))
    entropy135.append(entropyGlcm(Derajat135[i]))

    ASM0.append(ASM(Derajat0[i]))
    ASM45.append(ASM(Derajat45[i]))
    ASM90.append(ASM(Derajat90[i]))
    ASM135.append(ASM(Derajat135[i]))

    energy0.append(energy(Derajat0[i]))
    energy45.append(energy(Derajat45[i]))
    energy90.append(energy(Derajat90[i]))
    energy135.append(energy(Derajat135[i]))

    correlation0.append(correlation_feat(Derajat0[i]))
    correlation45.append(correlation_feat(Derajat45[i]))
    correlation90.append(correlation_feat(Derajat90[i]))
    correlation135.append(correlation_feat(Derajat135[i]))

print(f"Ekstraksi fitur tekstur (GLCM) selesai untuk {len(dataPreprocessedGray)} citra.")

## 7. Jalankan Ekstraksi Fitur Warna (Histogram HSV)

In [ ]:
Hue_feats = []
Sat_feats = []
Val_feats = []

for i in range(len(dataPreprocessedColor)):
    h, s, v = color_histogram_hsv(dataPreprocessedColor[i], bins=8)
    Hue_feats.append(h)
    Sat_feats.append(s)
    Val_feats.append(v)

Hue_feats = np.array(Hue_feats)
Sat_feats = np.array(Sat_feats)
Val_feats = np.array(Val_feats)

total_fitur_warna = Hue_feats.shape[1] + Sat_feats.shape[1] + Val_feats.shape[1]
print(f"Ekstraksi fitur warna (HSV histogram) selesai untuk {len(dataPreprocessedColor)} citra.")
print(f"Jumlah fitur warna per citra: {total_fitur_warna} "
      f"(Hue {Hue_feats.shape[1]} + Sat {Sat_feats.shape[1]} + Val {Val_feats.shape[1]})")

## 8. Gabungkan Fitur Tekstur + Warna jadi Satu Tabel

In [ ]:
dataTable = {
    'Filename': file_name, 'Label': labels,
    'Contrast0': Kontras0, 'Contrast45': Kontras45, 'Contrast90': Kontras90, 'Contrast135': Kontras135,
    'Homogeneity0': homogenity0, 'Homogeneity45': homogenity45, 'Homogeneity90': homogenity90, 'Homogeneity135': homogenity135,
    'Dissimilarity0': dissimilarity0, 'Dissimilarity45': dissimilarity45, 'Dissimilarity90': dissimilarity90, 'Dissimilarity135': dissimilarity135,
    'Entropy0': entropy0, 'Entropy45': entropy45, 'Entropy90': entropy90, 'Entropy135': entropy135,
    'ASM0': ASM0, 'ASM45': ASM45, 'ASM90': ASM90, 'ASM135': ASM135,
    'Energy0': energy0, 'Energy45': energy45, 'Energy90': energy90, 'Energy135': energy135,
    'Correlation0': correlation0, 'Correlation45': correlation45, 'Correlation90': correlation90, 'Correlation135': correlation135,
}

# Tambahin kolom fitur warna HSV (Hue0..7, Sat0..7, Val0..7)
for b in range(Hue_feats.shape[1]):
    dataTable[f'Hue{b}'] = Hue_feats[:, b]
for b in range(Sat_feats.shape[1]):
    dataTable[f'Sat{b}'] = Sat_feats[:, b]
for b in range(Val_feats.shape[1]):
    dataTable[f'Val{b}'] = Val_feats[:, b]

df = pd.DataFrame(dataTable)
df.to_csv('hasil_ekstraksi_pros4.csv', index=False)

hasilEkstrak = pd.read_csv('hasil_ekstraksi_pros4.csv')
hasilEkstrak

## 9. Seleksi Fitur (Korelasi)

Sama kayak Percobaan 1-3: kalau dua fitur korelasinya ≥ 0.95 (mirip banget / redundan), salah
satunya dibuang. Bedanya, sekarang fitur awalnya bukan cuma 28 (GLCM), tapi GLCM + warna —
jadi jumlahnya dihitung otomatis, nggak di-hardcode.

In [ ]:
corr_matrix = hasilEkstrak.drop(columns=['Label','Filename']).corr()

threshold = 0.95
columns = np.full((corr_matrix.shape[0],), True, dtype=bool)
for i in range(corr_matrix.shape[0]):
    for j in range(i+1, corr_matrix.shape[0]):
        if abs(corr_matrix.iloc[i,j]) >= threshold:
            if columns[j]:
                columns[j] = False

select = hasilEkstrak.drop(columns=['Label','Filename']).columns[columns]
x_new = hasilEkstrak[select]
y = hasilEkstrak['Label']

jumlah_fitur_awal = hasilEkstrak.drop(columns=['Label','Filename']).shape[1]
print(f"Fitur sebelum seleksi : {jumlah_fitur_awal}")
print(f"Fitur setelah seleksi : {len(select)}")
print(f"Fitur terpilih        : {list(select)}")

plt.figure(figsize=(20,20))
sns.heatmap(x_new.corr(), annot=True, cmap='Reds', fmt=".2f")
plt.title('Heatmap Korelasi Fitur - Percobaan 4', fontsize=14)
plt.show()

## 10. Split Data & Standardisasi

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_new, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape)
print(X_test.shape)

In [ ]:
mean_train = X_train.mean()
std_train  = X_train.std()

X_test  = (X_test  - mean_train) / std_train
X_train = (X_train - mean_train) / std_train

## 11. Definisikan Model Klasifikasi

In [ ]:
def generateClassificationReport(y_true, y_pred):
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))
    print('Accuracy:', accuracy_score(y_true, y_pred))

# Define classifiers
rf  = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=4, min_samples_leaf=2, max_features='sqrt', random_state=42)
svm = SVC(kernel='rbf', random_state=42, C=5, gamma='scale', class_weight='balanced')
knn = KNeighborsClassifier(n_neighbors=7, weights='distance')

In [ ]:
# Train Random Forest Classifier
rf.fit(X_train, y_train)

print("------Training Set------")
y_pred = rf.predict(X_train)
generateClassificationReport(y_train, y_pred)

print("\n------Testing Set------")
y_pred = rf.predict(X_test)
generateClassificationReport(y_test, y_pred)

In [ ]:
# Train SVM Classifier
svm.fit(X_train, y_train)

print("\n------Training Set------")
y_pred = svm.predict(X_train)
generateClassificationReport(y_train, y_pred)

print("\n------Testing Set------")
y_pred = svm.predict(X_test)
generateClassificationReport(y_test, y_pred)

In [ ]:
# Train KNN Classifier
knn.fit(X_train, y_train)

print("\n------Training Set------")
y_pred = knn.predict(X_train)
generateClassificationReport(y_train, y_pred)

print("\n------Testing Set------")
y_pred = knn.predict(X_test)
generateClassificationReport(y_test, y_pred)

## 12. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Reds)
    plt.title(title)
    plt.show()

# Plot confusion matrix for Random Forest
plot_confusion_matrix(y_test, rf.predict(X_test),  "Random Forest Confusion Matrix - Percobaan 4")
# Plot confusion matrix for SVM
plot_confusion_matrix(y_test, svm.predict(X_test), "SVM Confusion Matrix - Percobaan 4")
# Plot confusion matrix for KNN
plot_confusion_matrix(y_test, knn.predict(X_test), "KNN Confusion Matrix - Percobaan 4")

## 13. Ringkasan & Simpan Hasil Perbandingan Model

In [ ]:
# Simpan hasil perbandingan model
hasil_klasifikasi = {
    'Model'    : ['Random Forest', 'SVM', 'KNN'],
    'Accuracy_Train': [
        accuracy_score(y_train, rf.predict(X_train)),
        accuracy_score(y_train, svm.predict(X_train)),
        accuracy_score(y_train, knn.predict(X_train)),
    ],
    'Accuracy_Test': [
        accuracy_score(y_test, rf.predict(X_test)),
        accuracy_score(y_test, svm.predict(X_test)),
        accuracy_score(y_test, knn.predict(X_test)),
    ],
    'Precision': [
        precision_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        precision_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        precision_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
    'Recall': [
        recall_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        recall_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        recall_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
    'F1_Score': [
        f1_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        f1_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        f1_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
}
df_hasil = pd.DataFrame(hasil_klasifikasi)
df_hasil.to_csv('hasil_klasifikasi_pros4.csv', index=False)
print("✅ File hasil_klasifikasi_pros4.csv berhasil disimpan.")
df_hasil